# Assignment 4: Playing with Frequencies

MIDS W281: Computer Vision

## Recommended Libraries

In [ ]:
import os
from pathlib import Path

# Colab starts in /content and does not have the assignment images unless you clone/upload the repo.
if Path("images/al.png").exists():
    print("Using images in", Path("images").resolve())
elif Path("Assignment-4-repo/images/al.png").exists():
    os.chdir("Assignment-4-repo")
    print("Changed directory to", os.getcwd())
else:
    print("images/ not found. Cloning the assignment repo...")
    os.system("git clone https://github.com/parul-lakhotia/Assignment-4-repo.git")
    if Path("Assignment-4-repo/images/al.png").exists():
        os.chdir("Assignment-4-repo")
        print("Changed directory to", os.getcwd())
    else:
        raise FileNotFoundError(
            "Still cannot find images/al.png. In Colab: Runtime folder is /content. "
            "Either upload the images/ folder next to this notebook, or run:\n"
            "!git clone https://github.com/parul-lakhotia/Assignment-4-repo.git\n"
            "%cd Assignment-4-repo"
        )

print("cwd:", os.getcwd())
print("al.png exists:", Path("images/al.png").exists())


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import convolve

# After the Colab setup cell has cd'd into the repo, utils.py is in the current folder.
sys.path.insert(0, str(Path.cwd()))
from utils import create_2d_gaussian, normalize_img

print("create_2d_gaussian loaded from", Path("utils.py").resolve())


## Part 1: Hybrid Images

![Hybrid Teaser](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_4/objective1.png)

 ### Overview
 The goal of this part of the assignment is to create [hybrid images](http://olivalab.mit.edu/hybrid_gallery/gallery.html) using the approach described in the SIGGRAPH 2006 [paper](http://olivalab.mit.edu/publications/OlivaTorralb_Hybrid_Siggraph06.pdf) by Oliva, Torralba, and Schyns. Hybrid images are static images that change in interpretation as a function of the viewing distance. The basic idea is that high frequency tends to dominate perception when it is available, but, at a distance, only the low frequency part of the image can be seen. By blending the high-frequency portion of one image with the low-frequency portion of another, you get a hybrid image that leads to different interpretations at different distances.

 ### Description
 You are given a sample image of [Albert Einstein](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_3/al.png) and [Marlyn Monroe](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_3/mm.png). These images are already aligned such that the eyes are at the same location in the images.

 1. Low-pass the image of Albert Einstein. For a low-pass filter, Oliva et al. suggests using a standard 2-D Gaussian filter. The size and sigma of the Gaussian filter can be decided with some experimentation. In the given example, we used a Gaussian filter of size 15 and sigma 8. **Hint: You can use create_2d_gaussian(size, std) function in utils.py to create a 2-D Gaussian filter of a given size and sigma.**  

 2. High-pass filter the image of Marlyn Monroe. For a high-pass filter, the paper suggests using the impulse filter minus the Gaussian filter. This is the same method we used to create a high-pass filter [I - I<sub>b</sub>] in Part 1 of the previous assignment. For computing [I<sub>b</sub>] in this example, we used a Gaussian filter of size 9 and sigma 1.5. You can experiment with these values.  
 
 3. Create a hybrid image by taking the **average** of the output images in the above two steps.  

### Deliverables:

- Python code creating hybrid images
- The output hybrid image.

In [ ]:
# create a function that takes in an two images,
# uses the create_2d_gaussian method inside the utils script to create a 2D Gaussian filter
# then passes these filters to scipy.ndimage's convolve function
# finally, uses these blurred images to return a hybrid image

def hybrid_image(im1, im2):
    """im1 is low-pass (Einstein), im2 is high-pass (Monroe)."""
    from utils import create_2d_gaussian
    low_kernel = create_2d_gaussian(size=15, std=8)
    high_kernel = create_2d_gaussian(size=9, std=1.5)

    low_pass = convolve(im1, low_kernel, mode='reflect')
    high_pass = im2 - convolve(im2, high_kernel, mode='reflect')

    return 0.5 * (low_pass + high_pass)


In [ ]:
# load in the assignment images
imfile1 ='images/al.png'
imfile2 = 'images/mm.png'

im1 = plt.imread(imfile1) # low sf
im2 = plt.imread(imfile2) # high

# hybrid images
hybrid = hybrid_image(im1, im2)
hybrid_show = np.clip(hybrid, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(im1, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Einstein (low pass)')
axes[0].axis('off')
axes[1].imshow(im2, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Monroe (high pass)')
axes[1].axis('off')
axes[2].imshow(hybrid_show, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Hybrid')
axes[2].axis('off')
plt.tight_layout()
plt.show()

plt.imsave('out_hybrid.png', hybrid_show, cmap='gray', vmin=0, vmax=1)
print('Saved out_hybrid.png')


## Part 2: Multiresolution Blending (a.k.a. the oraple!)

![Multiresolution Teaser](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_4/objective2.png)

### Overview
The goal of this part of the assignment is to blend two images seamlessly using multi-resolution blending, as described in the 1983 [paper](http://persci.mit.edu/pub_pdfs/spline83.pdf) by Burt and Adelson. In this method, blending occurs at multiple scales so that low-frequency content is blended smoothly over a wide area, and high-frequency content is blended sharply over a narrow area. This is achieved using _image stacks_, which are similar to _image pyramids_. In an image pyramid, for each level of the pyramid the image is downsampled (so that each pyramid level gets smaller and smaller). In a stack, the images for each level are progressively blurred, but not downsampled, so that each pyramid level contains fewer high frequencies but is still of the same size as the original image. Images in a stack can all be saved in one 3D matrix if the original image was grayscale.

### Description

 1. Implement a function that creates Gaussian and Laplacian stacks. To create the successive levels of the Gaussian stack, just apply the Gaussian filter at each level, but do not downsample. The first image in a Gaussian stack is the original image. Therefore, the last image in an n-level stack is an image that is blurred n-1 number of times. For a Laplacian stack, take the difference between the successive levels of the Gaussian stacks. Additonally, the last level of a n-level Laplacian stack is an image blurred n-1 number of times (i.e., the remainder low-frequency information not included in the previous levels). In this project, you must implement your own Gaussian and Laplacian stacks. Do not use the pyrDown or pyrUp functions in Python.  
 ***Hint: Compare your final output image to the provided objective output.  If it does not look correct, consider how you calculate the difference between the successive levels of the Gaussian stack to calculate your Laplacian stack (which level is subtracted from another)***
 
 2. Display the six levels of the Laplacian stack in a single Python plot. Use the output hybrid image from Part 1 for creating the stacks in this deliverable. Notice the first level and the last level of the Laplacian stack. The first level (high frequency) will look like Marlyn Monroe and the last level (low frequency) will look like Albert Einstein. The size and sigma of the Gaussian kernel used for blurring can be the same as in Part 1.
 
 3. Now write code in order to use your Gaussian and Laplacian stacks for blending two images together. For this, we have included the two sample images from the paper (of an [apple](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_3/apple.png) and an [orange](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_3/orange.png)), and a [mask](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_3/mask.png) image. First, create Laplacian stacks for the two input images (six levels is sufficient). Additionally, create a Gaussian stack for your mask. In our example, we used a Gaussian kernel of size 17 and sigma 3. Display the stacks for the two input images and the mask.
 
 4. Create a blended image, B, using the Laplacian stacks of the two images (image X and image Y) and the Gaussian stack of the mask. The intensity of the blended image at location (i,j) can be computed using the following formula,
 
     $$B(i,j)=\sum_{l=1}^{n} [GM_l(i,j) \times LX_l(i,j) + (1 - GM_l(i,j)) \times LY_l(i,j)]$$ 
     
     where $GM_l$, $LX_l$, and $LY_l$ are the $l$ th level of: the Gaussian stack for the mask image, the Laplacian stacks of image X, and the Laplacian stacks of image Y and $n$ is the number of levels in the stack.
 
 5. Finally normalize all output images to visibly show their details (the mask won't look like it changes much, but the pixel values should still be different).


### Deliverables:

 - Python code for Gaussian and Laplacian stacks
 - Python code for image blending
 - A plot displaying the six levels of Laplacian stacks for the hybrid image
 - Plots displaying the six levels of Laplacian stacks for the two sample images
 - Plots displaying the six levels of Gaussian stacks for the mask image
 - The blended image, B


In [ ]:
# stack visualization
def visualize_stack(in_stack, title):
    n = len(in_stack)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.imshow(normalize_img(in_stack[i]), cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'{title} L{i}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# takes in a single channel
def gaussian_and_laplacian_stack(img, levels, size=17, std=3):
    kernel = create_2d_gaussian(size=size, std=std)
    gaussian_stack = [img.astype(np.float64)]
    for _ in range(levels - 1):
        gaussian_stack.append(convolve(gaussian_stack[-1], kernel, mode='reflect'))

    # L_i = G_i - G_{i+1}; last level keeps the leftover low frequencies
    laplacian_stack = [gaussian_stack[i] - gaussian_stack[i + 1] for i in range(levels - 1)]
    laplacian_stack.append(gaussian_stack[-1])
    return gaussian_stack, laplacian_stack

def collapse_laplacian_stack(laplacian_stack):
    return np.sum(np.stack(laplacian_stack, axis=0), axis=0)

def create_blended_stack(ls1, ls2, gr):
    blended = []
    for i in range(len(ls1)):
        blended.append(gr[i] * ls1[i] + (1 - gr[i]) * ls2[i])
    return blended

def multires_blending(img1, img2, mask, levels=6, size=17, std=3):
    _, ls1 = gaussian_and_laplacian_stack(img1, levels, size=size, std=std)
    _, ls2 = gaussian_and_laplacian_stack(img2, levels, size=size, std=std)
    gs_mask, _ = gaussian_and_laplacian_stack(mask, levels, size=size, std=std)
    blended_stack = create_blended_stack(ls1, ls2, gs_mask)
    return collapse_laplacian_stack(blended_stack)


### Deliverables

In [ ]:
img1 = plt.imread("images/apple.png")
img2 = plt.imread("images/orange.png")
mask = plt.imread("images/mask.png")

# create stacks for hybrid, apple, orange, mask
# hybrid stack uses the Part 1 blur (size 15, sigma 8)
gs_h, ls_h = gaussian_and_laplacian_stack(hybrid, 6, size=15, std=8)
gs_a, ls_a = gaussian_and_laplacian_stack(img1, 6)
gs_o, ls_o = gaussian_and_laplacian_stack(img2, 6)
gs_m, _ = gaussian_and_laplacian_stack(mask, 6)

# visualize all the stacks
visualize_stack(ls_h, title='laplacian hybrid')
visualize_stack(ls_a, title='laplacian apple')
visualize_stack(ls_o, title='laplacian orange')
visualize_stack(gs_m, title='gaussian mask')

blended = normalize_img(multires_blending(img1, img2, mask))

plt.figure(figsize=(4, 4))
plt.imshow(blended, cmap='gray', vmin=0, vmax=1)
plt.title('Oraple blend')
plt.axis('off')
plt.show()

plt.imsave("out_oraple.png", blended, cmap='gray', vmin=0, vmax=1)
print('Saved out_oraple.png')


## Acknowledgments
This assignment is inspired from Image Manipulation, Computer Vision and Computational Photography course at EECS UC Berkeley.